# scBeacon Quickstart

## Goal
Inspect three real prepared-input examples and learn how to invoke the shared tool.

**Mode:** executed frozen-result tutorial, not GPU training, DEG recomputation or live ToppGene. Read the user guide before launching an expensive run.

## Setup
Run on a compute node or your own workstation with requirements-notebooks installed. No large H5AD files or external API are needed for these default cells.

In [1]:
%matplotlib inline
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scbeacon.py").is_file())
sys.path.insert(0, str(ROOT / "01_tool/03_runtime"))
from results import summary, table, umap, network, heatmap, pathways


## Steps
### 1. Inspect the archived cohorts
Cell counts and clean-pair counts below are read from the actual frozen tables.

In [2]:
reports = [summary(d) for d in ("ipf", "covid", "lung")]
display(pd.DataFrame(reports)[["dataset", "n_cells", "selected_seed", "iteration", "clean_pairs"]])

,dataset,n_cells,selected_seed,iteration,clean_pairs
0,ipf,18030,23,33,9
1,covid,56485,4,33,20
2,lung,88144,38,33,23


### 2. Inspect one real pair result
The selected IPF cross-type population match is not a demonstrated lineage trajectory.

In [3]:
display(table("ipf", "clean_pairs.tsv")[["pair_id", "pair_cluster_label", "forward_fisher_qvalue_bh", "reverse_fisher_qvalue_bh"]])

,pair_id,pair_cluster_label,forward_fisher_qvalue_bh,reverse_fisher_qvalue_bh
0,B_pair_00,C3->D0,1.272727e-300,1.272727e-300
1,B_pair_01,C3->D1,1.272727e-300,1.272727e-300
2,B_pair_02,C7->D10,1.272727e-300,1.272727e-300
3,B_pair_04,C4->D12,1.272727e-300,1.272727e-300
4,B_pair_07,C6->D3,1.272727e-300,1.272727e-300
5,B_pair_09,C8->D5,1.272727e-300,1.272727e-300
6,B_pair_10,C8->D6,1.272727e-300,1.272727e-300
7,B_pair_11,C9->D7,1.272727e-300,1.272727e-300
8,B_pair_12,C3->D8,1.272727e-300,1.272727e-300


### 3. Configure a new run
The JSON template uses raw counts for training and a separately prepared normalized-log matrix for DEG. Relative paths resolve against the JSON location. Training resolution selection uses the supplied annotations.

In [4]:
import json
example = json.loads((ROOT / "01_tool/05_examples/user_pair.json").read_text())
display(pd.Series(example, name="configuration"))

dataset_id                                                user_pair
condition1_input              prepared/control_raw_top2000_hvg.h5ad
condition2_input              prepared/disease_raw_top2000_hvg.h5ad
combined_lognorm         prepared/combined_lognorm_top2000_hvg.h5ad
condition1_name                                             Control
condition2_name                                             Disease
label_key_1                                               cell_type
label_key_2                                               cell_type
seed                                                              0
iterations                                                       33
compatibility_profile                                       covid09
output_dir                                               runs/train
batch_size                                                      512
pretrain_epochs                                                  40
annotation_epochs                               

### 4. Preview before execution
Copy the JSON to your working directory and set real paths. The following are commands to run, not commands executed in this notebook. --execute is explicit; training requires GPU resources.

In [5]:
print("python scbeacon.py train --config /your/work/user_pair.json --output /your/work/train")
print("python scbeacon.py train --config /your/work/user_pair.json --output /your/work/train --execute")
print("python scbeacon.py plan --dataset covid --output /your/work/covid_run")

python scbeacon.py train --config /your/work/user_pair.json --output /your/work/train
python scbeacon.py train --config /your/work/user_pair.json --output /your/work/train --execute
python scbeacon.py plan --dataset covid --output /your/work/covid_run


## Checks
No training subprocesses, API requests or dataset modifications were performed.

In [6]:
assert [r["n_cells"] for r in reports] == [18030, 56485, 88144]
assert [r["clean_pairs"] for r in reports] == [9, 20, 23]
print("Frozen cohorts verified; training was not run.")

Frozen cohorts verified; training was not run.


## Next Steps
Open each dataset's executed notebook for UMAP, pairs, marker heatmap and selected pathway results. See 04_docs/02_reproduction.md for full scientific rerun interfaces and their validation boundary.